## Before Starting (If you haven't done it already)
Go to https://aistudio.google.com/app/apikey copy generative language client free tier API key

After you did that click the key icon at the left sidebar add your API key with GOOGLE_API_KEY as name and your API key as the value and enable notebook access

## Setup

### Install dependencies

In [1]:
%pip install -qU 'google-genai>=1.0.0'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 5.8 MB/s eta 0:00:00


### Set up your API key

To run the following cell, your API key must be stored it in a Colab Secret named `GOOGLE_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see the [Authentication](../quickstarts/Authentication.ipynb) quickstart for an example.

In [37]:
from google import genai
from google.colab import userdata
from google.genai import types

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=GOOGLE_API_KEY)

In [42]:
MODEL_ID = "gemini-2.5-flash-preview-tts" # @param ["gemini-2.5-flash-preview-tts","gemini-2.5-pro-preview-tts"] {"allow-input":true, isTemplate: true}

# How can the LLM *speak* to you

Now we need a way to let our LLM

Next create a helper function to prompt the model and play back the audio in the notebook:

In [4]:
# @title Helper functions (just run that cell)

import contextlib
import wave
from IPython.display import Audio

file_index = 0

@contextlib.contextmanager
def wave_file(filename, channels=1, rate=24000, sample_width=2):
    with wave.open(filename, "wb") as wf:
        wf.setnchannels(channels)
        wf.setsampwidth(sample_width)
        wf.setframerate(rate)
        yield wf

def play_audio_blob(blob):
  global file_index
  file_index += 1

  fname = f'audio_{file_index}.wav'
  with wave_file(fname) as wav:
    wav.writeframes(blob.data)

  return Audio(fname, autoplay=True)

def play_audio(response):
    return play_audio_blob(response.candidates[0].content.parts[0].inline_data)

## Generate a simple audio output

Let's start with something simple:

In [5]:
response = client.models.generate_content(
  model=MODEL_ID,
  contents="Say 'hello, my name is Gemini!'",
  config={"response_modalities": ['Audio']},
)

The generated ouput is in the response `inline_data` and as you can see it's indeed audio data.

In [6]:
blob = response.candidates[0].content.parts[0].inline_data
print(blob.mime_type)

audio/L16;codec=pcm;rate=24000


To be able to listen to the generated audio in colab, you're going to use our helper function to write the output in a file and play it.

In [7]:
play_audio_blob(blob)

Note that the model can only do TTS, so you should always tell it to "say", "read", "TTS" something, otherwise it won't do anything.

## Control how the model speaks

There are 30 different built-in voices you can use and 24 supported languages which gives you plenty of combinations to try.

### Choose a voice

Choose a voice among the 30 different ones. You can find their characteristics in the [documentation](https://ai.google.dev/gemini-api/docs/speech-generation#voices).

In [8]:
voice_name = "Sadaltager" # @param ["Zephyr", "Puck", "Charon", "Kore", "Fenrir", "Leda", "Orus", "Aoede", "Callirhoe", "Autonoe", "Enceladus", "Iapetus", "Umbriel", "Algieba", "Despina", "Erinome", "Algenib", "Rasalgethi", "Laomedeia", "Achernar", "Alnilam", "Schedar", "Gacrux", "Pulcherrima", "Achird", "Zubenelgenubi", "Vindemiatrix", "Sadachbia", "Sadaltager", "Sulafar"]

In [9]:
response = client.models.generate_content(
  model=MODEL_ID,
  contents="""Say "I am a very knowlegeable model, especially when using grounding", wait 5 seconds then say "Don't you think?".""",
  config={
      "response_modalities": ['Audio'],
      "speech_config": {
          "voice_config": {
              "prebuilt_voice_config": {
                  "voice_name": voice_name
              }
          }
      }
  },
)

play_audio(response)

### Change the language

Just tell the model to speak in a certain language and it will. The [documentation](https://ai.google.dev/gemini-api/docs/speech-generation#languages) lists all the supported ones.

In [10]:
response = client.models.generate_content(
  model=MODEL_ID,
  contents="""
    Read this in French:

    Les chaussettes de l'archiduchesse sont-elles sèches ? Archi-sèches ?
    Un chasseur sachant chasser doit savoir chasser sans son chien.
  """,
  config={"response_modalities": ['Audio']},
)

play_audio(response)

In [18]:
response = client.models.generate_content(
  model=MODEL_ID,
  contents="""
    Read this in Turkish in a worried tone wait 3 seconds after saying "Tam kurudu mu yani?":

    Arşidüşesin çorapları kuru mu? Tam kurudu mu yani?  Kurudu yani eminsin di mi?
  """,
  config={"response_modalities": ['Audio']},
)

play_audio(response)

### Prompt the model to speak in certain ways

You can control style, tone, accent, and pace using natural language prompts, for example:

In [19]:
response = client.models.generate_content(
  model=MODEL_ID,
  contents="""
    Say in an spooky whisper:
    "By the pricking of my thumbs...
    Something wicked this way comes!"
  """,
  config={"response_modalities": ['Audio']},
)

play_audio(response)

In [25]:
response = client.models.generate_content(
  model=MODEL_ID,
  contents="""
    Read this disclaimer in as fast a voice as possible in Turkish while remaining intelligible:

    Yazar bu sitedeki içerikte yer alan herhangi bir hata veya eksiklikten dolayı hiçbir sorumluluk veya yükümlülük kabul etmez.
    Bu sitede yer alan bilgiler, eksiksizlik, doğruluk, fayda veya güncellik konusunda hiçbir garanti verilmeden, “olduğu gibi” sağlanmaktadır.
  """,
  config={"response_modalities": ['Audio']},
)

play_audio(response)

## Multi-speakers

The TTS model can also read discussions between 2 speakers. You just need to tell it that there are two speakers:

In [39]:
response = client.models.generate_content(
  model=MODEL_ID,
  contents="""
    Make Speaker1 sound tired and bored, and Speaker2 sound excited and happy:

    Speaker1: Yine ne oldu da çağırdın beni?
    Speaker2: Asla tahmin edemiyeceksin muhteşem bişey!
  """,
  config={"response_modalities": ['Audio']},
)

play_audio(response)

You can also select the voices for each participants and pass their names to the model.

But first let's generate a discussion between two scientists:

In [40]:
transcript = client.models.generate_content(
    model='gemini-2.5-flash',
    contents="""
      Hi, please generate a short (like 100 words) transcript that reads like
      it was clipped from a podcast by excited herpetologists, Dr. Claire and
      her assistant, the young Aurora.
    """
  ).text

print(transcript)

*(Sound of crickets chirping softly in background, then fades slightly)*

**Dr. Claire:** ...and honestly, Aurora, seeing that *Theloderma corticale* in its natural habitat? Absolutely breathtaking.

**Aurora:** Oh my gosh, Dr. Claire, I still get shivers! The way it just… *disappeared*! Its camouflage isn't just color, it's those incredible tuberculated glands mimicking moss and lichen *perfectly*.

**Dr. Claire:** Precisely! It's not just blending in, it's *becoming* the environment. A true master of cryptic coloration. You think you're looking at a rock, then suddenly, *blink!* Those enormous, watchful eyes.

**Aurora:** I swear, I nearly stepped on it twice before I even saw it move! The Mossy Frog lives up to its name and then some. It’s just… incredible.

**Dr. Claire:** Evolutionary marvel, isn't it? Just phenomenal.


Then let's have the TTS model render the conversation using the voices you want.

In [41]:
config = types.GenerateContentConfig(
    response_modalities=["AUDIO"],
    speech_config=types.SpeechConfig(
        multi_speaker_voice_config=types.MultiSpeakerVoiceConfig(
            speaker_voice_configs=[
                types.SpeakerVoiceConfig(
                    speaker='Dr. Claire',
                    voice_config=types.VoiceConfig(
                        prebuilt_voice_config=types.PrebuiltVoiceConfig(
                            voice_name='sulafat',
                        )
                    )
                ),
                types.SpeakerVoiceConfig(
                    speaker='Aurora',
                    voice_config=types.VoiceConfig(
                        prebuilt_voice_config=types.PrebuiltVoiceConfig(
                            voice_name='Leda',
                        )
                    )
                ),
            ]
        )
    )
)

response = client.models.generate_content(
  model=MODEL_ID,
  contents="TTS the following conversation between a very excited Dr. Claire and her assistant, the young Aurora: "+transcript,
  config=config,
)

play_audio(response)

That's it for our second notebook.
We will continue at https://aistudio.google.com/app/live
